# Project - Airline AI Assistant.


In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:8]}")
else:
    print("API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-a


In [53]:
system_message += (
    " When you look up a ticket price, check the 'status' field in the result. "
    "If the status is 'city_not_found', apologise briefly and say you don't have fares for that destination. "
    "If it is 'class_not_found', say that class isn't available and offer the classes in 'available_classes'. "
    "Never make up a price."
)

In [27]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Optionally specify the travel class. If no class is given, prices for all available classes are returned.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "ticket_class": {
                "type": "string",
                "description": "The travel class, e.g. economy, business, first",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

In [45]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('DROP TABLE IF EXISTS prices')
    cursor.execute('''
        CREATE TABLE prices (
            city TEXT,
            ticket_class TEXT,
            price REAL,
            PRIMARY KEY (city, ticket_class)
        )
    ''')
    conn.commit()

In [ ]:
def get_ticket_price(city, ticket_class=None):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        if ticket_class:
            cursor.execute(
                'SELECT price FROM prices WHERE city = ? AND ticket_class = ?',
                (city.lower(), ticket_class.lower())
            )
            row = cursor.fetchone()
            return row[0] if row else None
        cursor.execute(
            'SELECT ticket_class, price FROM prices WHERE city = ?',
            (city.lower(),)
        )
        return dict(cursor.fetchall()) or None

In [47]:
def set_ticket_price(city, price, ticket_class):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO prices (city, ticket_class, price) VALUES (?, ?, ?) '
            'ON CONFLICT(city, ticket_class) DO UPDATE SET price = excluded.price',
            (city.lower(), ticket_class.lower(), price)
        )
        conn.commit()

In [48]:
ticket_prices = {
    "delhi": {"economy": 799, "business": 1899, "first": 3499},
    "chennai": {"economy": 899, "business": 2199, "first": 3999},
    "hyderabad": {"economy": 1400, "business": 3200, "first": 5500},
    "bangalore": {"economy": 499, "business": 1299},
}

for city, classes in ticket_prices.items():
    for ticket_class, price in classes.items():
        set_ticket_price(city, price, ticket_class)
    

In [50]:
get_ticket_price("delhi")

DATABASE TOOL CALLED: Getting price for delhi


{'business': 1899.0, 'economy': 799.0, 'first': 3499.0}

In [52]:
def get_available_classes(city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT ticket_class FROM prices WHERE city = ?', (city.lower(),))
        return [row[0] for row in cursor.fetchall()]

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            try:
                arguments = json.loads(tool_call.function.arguments)
                city = arguments.get("destination_city")
                ticket_class = arguments.get("ticket_class")

                if not city:
                    result = {"status": "error", "message": "No destination city was provided."}
                else:
                    price_details = get_ticket_price(city, ticket_class)
                    if price_details is None:
                        available = get_available_classes(city)
                        if available:
                            result = {
                                "status": "class_not_found",
                                "destination_city": city,
                                "ticket_class": ticket_class,
                                "available_classes": available,
                                "message": f"No {ticket_class} fare for {city}.",
                            }
                        else:
                            result = {
                                "status": "city_not_found",
                                "destination_city": city,
                                "message": f"No fares available for {city}.",
                            }
                    else:
                        result = {
                            "status": "ok",
                            "destination_city": city,
                            "ticket_class": ticket_class,
                            "price": price_details,
                        }
            except Exception as e:
                result = {"status": "error", "message": f"Lookup failed: {e}"}
        else:
            result = {"status": "error", "message": f"Unknown tool: {tool_call.function.name}"}

        responses.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id,
        })
    return responses

In [55]:
def chat(message,history):
    history = [{"role": h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model=MODEL,messages=messages,tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses =handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages,tools=tools)
    return response.choices[0].message.content

DATABASE TOOL CALLED: Getting price for delhi
DATABASE TOOL CALLED: Getting price for chennai
DATABASE TOOL CALLED: Getting price for chennai
DATABASE TOOL CALLED: Getting price for chennai


In [41]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for delhi
